In [ ]:
import gc
import os
import sys
from dataclasses import dataclass
# from google.colab import userdata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pprint import pprint
from typing import Callable
from datetime import datetime

import random
# from tqdm import tqdm
import time
from tqdm.autonotebook import tqdm as notebook_tqdm
import torch
from torch.utils.data import Dataset, DataLoader
# torch.set_float32_matmul_precision('high')

from huggingface_hub import login
import transformers
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM, AutoConfig, logging

import warnings
warnings.filterwarnings('ignore')

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
gc.collect()
torch.cuda.empty_cache()

In [ ]:
sys.version

In [ ]:
class Dataset10x(Dataset):
    def __init__(self, report_df: pd.DataFrame, items_path: str = 'items') -> None:
        super().__init__()

        self.target_columns = ['FILING_DATE', 'CIK', 'ACC_NUM']
        for col in self.target_columns:
            assert col in report_df.columns, col

        self.name_columns = [self.target_columns[0], '10-K_edgar_data'] + self.target_columns[1:]
        self.item_names = ['item1', 'item1a', 'item7']
        self.items_path = items_path
        self.report_df = report_df

    def __len__(self) -> int:
        return len(self.report_df)

    def __getitem__(self, idx: int) -> dict[str, str]:
        report_dict = {}
        notnone = False
        report = self.report_df.iloc[idx].to_dict()
        report_name = '_'.join([str(report[x]) if x in report.keys() else x for x in self.name_columns])

        for item_name in self.item_names:
            item_pathname = os.path.join(self.items_path, f'{item_name}_files', f'{report_name}_{item_name}.txt')

            if not os.path.exists(item_pathname):
                item_pathname = os.path.join(self.items_path, f'{report_name}_{item_name}.txt')

            if os.path.exists(item_pathname):
                with open(item_pathname, 'r', encoding='utf-8') as file:
                    item_text = file.read()
                    report_dict[item_name] = item_text
                    notnone = True
        return (idx, report_dict) if notnone else (idx, None)


def collate_fn_filter_none(batch):
    filtered = [(idx, data) for idx, data in batch if data is not None]
    if not filtered:
        return None
    indices, data = zip(*filtered)
    return list(indices), list(data)


# def collate_fn_item7(batch):
#     filtered = [(idx, data["item7"]) for idx, data in batch
#                 if data is not None and "item7" in data]
#     if not filtered:
#         return None
#     indices, item7_texts = zip(*filtered)
#     return list(indices), list(item7_texts)

def collate_fn_item7(batch):
    indices = []
    item7_texts = []
    
    for idx, data in batch:
        if data is not None and "item7" in data:
            item7 = data["item7"]
#             if len(item7) > 200:
            indices.append(idx)
            item7_texts.append(item7)
    return indices, item7_texts

In [ ]:
def fill_prompt_batch(texts, ending, prompt_func, tokenizer, model,
                      top_k=1, top_p=None, text_length=1000, verbose=False, device='cpu'):
    """
    - Generates tokens for a batch of texts
    - Uses model to obtain the logits (prob to fill mask)
    - Returns the top_k tokens and associated probs
    """
    clean_mem()

    prompt_texts = [prompt_func(text[:text_length], ending) for text in texts]

    # Encode prompts
    inputs = tokenizer(prompt_texts, return_tensors="pt", padding=True).to(device)
    if verbose:
        print(inputs['input_ids'].shape)
    # Forward pass
    with torch.no_grad():
        logits = model(**inputs).logits.cpu()
    # Get probs for last token only
    probss = torch.nn.functional.softmax(logits[:,-1,:], dim=-1)

    del inputs
    gc.collect()

    if top_p is not None:
        sorted_probs, sorted_indices = torch.sort(probss, descending=True, dim=-1)    # batch by V
        cumulative_probs = torch.cumsum(sorted_probs, dim=-1)                         # batch by V
        # Remove tokens with cumulative probability above the threshold
        sorted_indices_to_keep = cumulative_probs <= top_p                           # top_p
        # Shift the indices to the right to keep also the first token above the threshold
        sorted_indices_to_keep[..., 1:] = sorted_indices_to_keep[..., :-1].clone()
        sorted_indices_to_keep[..., 0] = True
        indices_to_keep = [sorted_indices[i][sorted_indices_to_keep[i]].tolist() for i in range(len(sorted_indices))]
        probs_to_keep = [sorted_probs[i][sorted_indices_to_keep[i]].tolist()  for i in range(len(sorted_indices))]
        answers = [dict(zip(l.strip().split(),probs_to_keep[i]))
                for i,l in enumerate(tokenizer.batch_decode(indices_to_keep))]

    else:
        # top k
        top_k_tokens = [torch.topk(probss[i], top_k, dim=0) for i in range(len(probss))]
        top_decoded = [list(map(lambda s: s.strip(),
                        tokenizer.batch_decode(top_k_tokens[i].indices))) for i in range(len(top_k_tokens))]
        top_prob = [top_k_tokens[i].values.tolist() for i in range(len(top_k_tokens))]

        answers = [dict(zip(top_decoded[i], top_prob[i])) for i in range(len(top_decoded))]

    return answers


In [ ]:
def apply_strategy(texts, ending, strategy, tokenizer, model, text_length=5000, verbose=False, device='cpu'):
    '''
    Applies prompt strategy

    - text: to be appended the prompt
    - strategy: contains the prompt and the verbalizer
    '''

    output = fill_prompt_batch(texts, ending, strategy.prompt, tokenizer=tokenizer,
                               model=model, top_p=strategy.top_p, text_length=text_length, device=device)
    clean_mem()

    scores = []
    for item in output:
        score = dict()
        for cat, vals in strategy.verbalizer.items():
            # Strip spaces + turn lowercase, then match with verbalizer
            score[cat] = sum([v for k,v in item.items() if k.strip().lower() in vals])
            try:
                score['polarity'] = score['positive'] - score['negative']
            except Exception:
                pass
        scores.append(score)

    if verbose:
        print(scores)
    return scores

In [ ]:
def get_model_ouput(prompt: str, model, device) -> dict[str, float]:
#     device = 'cuda:0,1,2,3'
    input = tokenizer(prompt, return_tensors='pt', padding=True).to(device)
    # print(input)
    print(f"input has {input['input_ids'].shape} tokens")

    with torch.no_grad():
        logits = model(**input).logits.cpu()

    logits = logits[0,-1,:]
    probs = torch.nn.functional.softmax(logits, dim=-1)
    top_k_tokens = torch.topk(probs, 20, dim=0)
    top_ix = top_k_tokens.indices.tolist()
    top_prob = top_k_tokens.values.tolist()

    # return dict(zip(top_ix, top_prob))
    return dict(zip([tokenizer.decode([x]) for x in top_ix], top_prob))

In [ ]:
def clean_mem():
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    time.sleep(0.1)

In [ ]:
def gather_stats(strategy, results, tokenizer, model, data, ending, verbose=False, text_length=5000, 
                save_path="results", save_interval=5000, resume=True, max_retries=3, device='cpu'):
    
    if resume and results:
        max_processed_idx = max([df.index.max() for df in results if not df.empty])
        print(f"Resuming from index: {max_processed_idx}")
    else:
        max_processed_idx = -1
#         print("Starting from beginning")
    
    processed_count = 0
    batch_count = 0
    
    for indices, batch in notebook_tqdm(data):
        if len(indices) > 0:
            if resume and max(indices) <= max_processed_idx:
                continue
            else:
                success = False
                retry_count = 0

                while not success and retry_count < max_retries:
                    try:
                        current_text_length = text_length // (2 ** retry_count) if retry_count > 0 else text_length

                        if retry_count > 0:
                            print(f"Retry {retry_count} for indices {indices} with text_length={current_text_length}")

                        ans = apply_strategy(batch, ending, strategy=strategy, tokenizer=tokenizer,
                                            model=model, text_length=current_text_length, device=device)
                        clean_mem()
                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()

                        df = pd.DataFrame(ans, index=indices)
                        results.append(df)
                        processed_count += len(indices)
                        batch_count += 1

                        if verbose:
                            print(f"Processed indices: {indices}")
                            print(f"Total processed: {processed_count}")

                        success = True

                    except torch.cuda.OutOfMemoryError as oom_error:
                        retry_count += 1
                        print(f"CUDA OOM error processing batch {indices} (attempt {retry_count}): {oom_error}")

                        clean_mem()
                        if torch.cuda.is_available():
                            torch.cuda.empty_cache()

                        if retry_count >= max_retries:
                            print(f"Failed to process batch {indices} after {max_retries} attempts")
                            
                            error_file = os.path.join(save_path, f"error_batch_{indices[0]}_{indices[-1]}.txt")
                            with open(error_file, 'w') as f:
                                f.write(f"Batch indices: {indices}\n")
                                f.write(f"Error: CUDA Out of Memory after {max_retries} retries\n")
                                f.write(f"Original text length: {text_length}\n")
                                f.write(f"Final attempted text length: {current_text_length}\n")
                            break

                    except Exception as e:
                        print(f"Error processing batch with indices {indices}: {e}")
                        error_file = os.path.join(save_path, f"error_batch_{indices[0]}_{indices[-1]}.txt")
                        with open(error_file, 'w') as f:
                            f.write(f"Batch indices: {indices}\n")
                            f.write(f"Error: {str(e)}\n")
                        break

                if processed_count >= save_interval:
                    try:
                        stats_df = pd.concat(results, axis=0)
                        stats_df.reset_index(inplace=True)

                        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
                        csv_file = os.path.join(save_path, f"results_{timestamp}_{processed_count}_reports.csv")
                        stats_df.to_csv(csv_file, index=False)
#                         print(f"Saved CSV: {csv_file}")

                        processed_count = 0

                    except Exception as e:
                        print(f"Error saving intermediate results: {e}")

        else:
            continue
    
    if results:
        try:
            stats_df = pd.concat(results, axis=0)
            stats_df.reset_index(inplace=True)
            
            final_csv_file = os.path.join(save_path, "final_results.csv")
            stats_df.to_csv(final_csv_file, index=False)
#             print(f"Saved final CSV: {final_csv_file}")
            
            if verbose:
                print(stats_df.info())
            
            return stats_df
        
        except Exception as e:
            print(f"Error saving final results: {e}")
            for i, df in enumerate(results):
                try:
                    backup_file = os.path.join(save_path, f"backup_result_{i}.pkl")
                    df.to_pickle(backup_file)
                except:
                    pass
            return pd.DataFrame()
    else:
        print("No results to process")
        return pd.DataFrame()

In [ ]:
def calculate_mean_polarities(tech_res, blue_chip_res):
    result_dict = {}
    tech_sent_before_crisis = tech_res[tech_res['FILING_DATE'].astype(str).str[:4].between('1997', '1999')].reset_index(drop=True)
    tech_sent_after_crisis = tech_res[tech_res['FILING_DATE'].astype(str).str[:4].between('2000', '2003')].reset_index(drop=True)

    blue_chip_before_crisis = blue_chip_res[blue_chip_res['FILING_DATE'].astype(str).str[:4].between('1997', '1999')].reset_index(drop=True)
    blue_chip_after_crisis = blue_chip_res[blue_chip_res['FILING_DATE'].astype(str).str[:4].between('2000', '2003')].reset_index(drop=True)
    
    tech_line = [tech_sent_before_crisis['polarity'].mean(), tech_sent_after_crisis['polarity'].mean()]
    result_dict['tech'] = {}
    result_dict['tech']['before'] = tech_line[0]
    result_dict['tech']['after'] = tech_line[1]
    delta = tech_line[0] - tech_line[1]
    
#     print(f'Tech companies before crisis: {round(tech_line[0], 3)}, after the crisis: {round(tech_line[1], 3)}')
#     print('Tech delta', round(tech_line[0] - tech_line[1], 3))
#     print('='*65)

    blue_chip_line = [blue_chip_before_crisis['polarity'].mean(), blue_chip_after_crisis['polarity'].mean()]
    delta = blue_chip_line[0] - blue_chip_line[1]
    result_dict['blue_chip'] = {}
    result_dict['blue_chip']['before'] = blue_chip_line[0]
    result_dict['blue_chip']['after'] = blue_chip_line[1]
    
#     print(f'Blue chip companies before crisis: {round(blue_chip_line[0], 3)}, after the crisis: {round(blue_chip_line[1], 3)}')
#     print('Blue chip delta', round(blue_chip_line[0] - blue_chip_line[1], 3))
#     print('='*65)

    res_df = pd.DataFrame.from_dict(result_dict).T
    res_df['delta'] = res_df['before'] - res_df['after']
    res_df = round(res_df, 4)
    display(res_df)
    
    plt.title('Polarity general Llama', weight='bold')
    plt.plot(tech_line, marker='o', label='tech')
    plt.plot(blue_chip_line, marker='o', label='non-tech')

    # for i in range(len(tech_line)):
    #     plt.text(i, tech_line[i], f'  {tech_line[i]:.3f}', va='bottom', ha='left', fontsize=9)
    #     plt.text(i, blue_chip_line[i], f'  {blue_chip_line[i]:.3f}', va='bottom', ha='left', fontsize=9)

    plt.ylabel('mean sentiment')
    plt.xticks(ticks=[0, 1], labels=['before crisis (1997 -- 1999)', 'after crisis (2001 -- 2003)'])
    plt.legend()
    plt.grid()

    return tech_line, blue_chip_line, result_dict

def calc_tech_and_blue_chip_sent(model, blue_chip_dataloader, tech_dataloader, ending, sentiment_verb):
    sentiment_strategy = Prompt_Strategy('sentiment', sentiment_verb, get_prompt)
    
    results = []

    blue_chip_sent = gather_stats(
        sentiment_strategy, results=results, tokenizer=tokenizer, model=model,
        data=blue_chip_dataloader, ending=ending, verbose=False, text_length=q95, 
        save_path="/home/jovyan/datavol-2/speculative_sent/blue_chip_sentiment/",
        save_interval=2000, resume=True, max_retries=3
    )

    blue_chip_res = blue_chip_df.join(blue_chip_sent)
    clean_mem()
    
    results = []

    tech_sent = gather_stats(
        sentiment_strategy, results=results, tokenizer=tokenizer, model=model,
        data=tech_dataloader, ending=ending, verbose=False, text_length=q95, 
        save_path="/home/jovyan/datavol-2/speculative_sent/tech_sentiment/",
        save_interval=2000, resume=True, max_retries=3
    )

    tech_res = tech_df.join(tech_sent)
    clean_mem()
    
    return tech_res, blue_chip_res

# Data

In [ ]:
df = pd.read_sas('/home/jovyan/datavol-2/finaldata_10k.sas7bdat')

df['ACC_NUM'] = df['filename'].astype(str).str.split('_').str[1]
df['CIK'] = df['filename'].astype(str).str.split('_').str[0]
df['FILING_DATE'] = df['date_filed'].astype(str).str.replace('-', '')
reports = df
reports
# reports = df[df['FILING_DATE'].astype(str).str[:4] < '2013'].reset_index(drop=True)
# reports = reports[['CIK', 'FILING_DATE', 'ACC_NUM']]
# reports

# df = pd.read_csv('/home/jovyan/datavol-2/missing_item7.tsv.gz', sep='\t')
# df['ACC_NUM'] = df['name'].astype(str).str.split('_').str[-1]
# df['CIK'] = df['name'].astype(str).str.split('_').str[-2]
# df['FILING_DATE'] = df['name'].astype(str).str.split('_').str[0]
# reports = df[['CIK', 'FILING_DATE', 'ACC_NUM']]
# reports

In [ ]:
summary = pd.read_csv('/home/jovyan/datavol-2/scripts/models/Loughran-McDonald_10X_Summaries_1993-2023.csv')

In [ ]:
tech_names = ['CISCO', 'AMAZON ', 'ETOYS', 'INTEL ', 'NETSCAPE', 'PETS ', 'NETSCAPE', 'YAHOO', 'FLOOZ', 'GEEKNET',
            'INTERACTIVE INTELLIGENCE', 'NETWORK SOLUTIONS', 'PALM', 'PIXELON', 'QUALCOMM', 'STEEL CONNECT', 'TRANSMETA',
            'UUNET', 'VERIO', 'TIBCO', 'UBID', 'WEBVAN']

blue_chip_names = ['COCA COLA', 'AT&T', 'PHILIP MORRIS', 'DUPONT', 'MERCK', 'GENERAL ELECTRIC']

selected_columns = ['CIK', "FILING_DATE", 'ACC_NUM', 'CoName']

tech_df = summary[summary['CoName'].str.startswith(tuple(tech_names))][selected_columns].reset_index(drop=True)
tech_df = tech_df[tech_df['FILING_DATE'].astype(str).str[:4].between('1997', '2003')].reset_index(drop=True)

blue_chip_df = summary[summary['CoName'].str.startswith(tuple(blue_chip_names))][selected_columns].reset_index(drop=True)
blue_chip_df = blue_chip_df[blue_chip_df['FILING_DATE'].astype(str).str[:4].between('1997', '2003')].reset_index(drop=True)

In [ ]:
items_path = '/home/jovyan/datavol-2/items/item7_files'
# items_path = '/home/jovyan/datavol-2/missing_item7'

tech_dataset = Dataset10x(tech_df, items_path)

not_none = []
for idx in range(393):
    item_dict = tech_dataset[idx]
    if item_dict[1] is not None:
        not_none.append(idx)
        
len(not_none)

In [ ]:
tech_df = tech_df.iloc[not_none].reset_index(drop=True)

In [ ]:
blue_chip_dataset = Dataset10x(blue_chip_df, items_path)

not_none = []
for idx in range(402):
    item_dict = blue_chip_dataset[idx]
    if item_dict[1] is not None:
        not_none.append(idx)
        
len(not_none)

In [ ]:
blue_chip_df = blue_chip_df.iloc[not_none].reset_index(drop=True)

In [ ]:
blue_chip_df.shape, tech_df.shape

In [ ]:
tech_df['FILING_DATE'].astype(str).str[:4].value_counts().sort_index()

In [ ]:
blue_chip_df['FILING_DATE'].astype(str).str[:4].value_counts().sort_index()

In [ ]:
batch_size = 2

blue_chip_dataset = Dataset10x(blue_chip_df, items_path)
blue_chip_dataloader = DataLoader(blue_chip_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn_item7)

tech_dataset = Dataset10x(tech_df, items_path)
tech_dataloader = DataLoader(tech_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn_item7)

# Model - Llama 3.1

In [ ]:
version = '3.1'
# Context length = 128k

if version=='3':
    model_id = "meta-llama/Meta-Llama-3-8B"
elif version=='3.1':
#     model_id = "meta-llama/Meta-Llama-3.1-8B"
    model_id = '/home/jovyan/models-2/Meta-Llama-3.1-8B'
elif version=='3.2':
    model_id = "meta-llama/Llama-3.2-3B"
elif version=='3.2-1b':
    model_id = "meta-llama/Llama-3.2-1B"
else:
    raise Exception("Set correct version")

print(f"Model {version=} -- {model_id=}")
logging.set_verbosity_error()

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(model_id, cache_dir="hf_cache")
if not tokenizer.pad_token:
    print("Adding pad_token")
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"
# vocab = set([*tokenizer.get_vocab()])

model = AutoModelForCausalLM.from_pretrained(
        model_id, torch_dtype=torch.bfloat16, cache_dir="hf_cache"
)

model = torch.nn.DataParallel(model)
model.to(device)
model.name = 'Llama3_1'

In [ ]:
clean_mem()

In [ ]:
item_name = 'item7'

q99 = 275568
q95 = 169981

## Prompt strategy

In [ ]:
@dataclass(frozen=True)
class Prompt_Strategy:
    name: str
    verbalizer: dict
    prompt: Callable
    top_p: float = 0.9
        
def get_prompt(item_text: str, ending: str) -> str:
    prompt = f"{item_text}\n{ending}"
    return prompt

sentiment_verb = {
    "positive": set(['good', 'best', 'excellent', 'outstanding', 'exceptional',
                     'healthy', 'strong', 'awesome', 'great', 'fantastic', 'stable',
                     'perfect', 'solid', 'profitable', 'impressive', 'reliable',
                     'thriving', 'optimistic', 'decent', 'positive', 'sustainable',]),
    
    "negative": set(['bad', 'poor', 'terrible', 'risky', 'weak', 'dependent',
                     'unstable', 'unhealthy', 'questionable', 'suffering', 'stressed',
                     'fragile', 'negative', 'unsustainable', 'awful', 'vulnerable',
                     'mediocre', 'horrible', 'precarious', 'declining', 'worsening'])
}

## Report probabilities

In [ ]:
clean_mem()

In [ ]:
idx = 34

item_dict = blue_chip_dataset[idx]
item_text = item_dict[1].get(item_name)

date = blue_chip_df.iloc[idx].FILING_DATE
co_name = blue_chip_df.iloc[idx]['CoName']

print(f"{co_name}, date: {date}")

In [ ]:
# pprint(item_text)

In [ ]:
# ending = "In one word, the company’s financial health is"
# ending = "Our outlook for the next fiscal year is fundamentally"
# ending = "In one word, our forward-looking statements are"
# ending = "The overall tone of this report regarding the future can be described as"
ending = "Based on current trends, we view the company's near-term trajectory with"

prompt = get_prompt(item_text, ending)
probs = get_model_ouput(prompt, model, device)
probs

In [ ]:
clean_mem()

## Prompts

In [ ]:
ending = "In one word, the company’s financial health is"

tech_res, blue_chip_res = calc_tech_and_blue_chip_sent(model, blue_chip_dataloader, tech_dataloader, ending)
tech_line, blue_chip_line, result_dict = calculate_mean_polarities(tech_res, blue_chip_res)

In [ ]:
ending = "In one word, the company's next period profit is going to be"

tech_res, blue_chip_res = calc_tech_and_blue_chip_sent(model, blue_chip_dataloader, tech_dataloader, ending)
tech_line, blue_chip_line, result_dict = calculate_mean_polarities(tech_res, blue_chip_res)

In [ ]:
ending = "Regarding future growth, we are"

sentiment_verb = {
    "positive": set(['optimistic', 'confident', 'positive', 'encouraged', 'excited',
                     'enthusiastic', 'hopeful', 'comfortable', 'satisfied', 'pleased', 
                     'relaxed', 'encouraged', 'ambitious', 'pleased', 'favorable', 
                     'assured', 'strong', 
             
                     ]),
    
    "negative": set(['cautious', 'concerned', 'aggressive', 'pessimistic', 'negative',
                     'uncomfortable', 'concerned', 'cautious', 'uncertain', 'unsure',
                     'skeptical', 'worried', 'discouraged', 'anxious', 'frustrated',
                     'confused', 'doubtful', 'unsatisfied', 'disappointed'
                    ])
}

tech_res, blue_chip_res = calc_tech_and_blue_chip_sent(model, blue_chip_dataloader, tech_dataloader, ending, sentiment_verb)
tech_line, blue_chip_line, result_dict = calculate_mean_polarities(tech_res, blue_chip_res)

In [ ]:
# ok
ending = "Future growth prospects appear"

tech_res, blue_chip_res = calc_tech_and_blue_chip_sent(model, blue_chip_dataloader, tech_dataloader, ending, sentiment_verb)
tech_line, blue_chip_line, result_dict = calculate_mean_polarities(tech_res, blue_chip_res)

In [ ]:
ending = "The overall tone of this report regarding the future can be described as"

sentiment_verb = {
    "positive": {
        "confident", "optimistic", "bullish", "upbeat", "positive",
        "encouraging", "hopeful", "reassuring", "constructive",
        "assured", "decisive", "resolute", "convincing", "definitive",
        "authoritative", "forceful", "assertive", "ambitious", "expansive",
        "growth-oriented", "forward-looking",
        "stable", "steady", "consistent", "dependable", "resilient",
        "robust", "solid", "secure"
    }, 
    "negative": {
        "cautious", "guarded", "prudent", "conservative", "hesitant",
        "uncertain", "vague", "noncommittal", "ambiguous", "qualified",
        "measured", "tempered", "subdued",
        "pessimistic", "bearish", "negative", "concerned", "apprehensive",
        "worrisome", "troubling", "alarming", "ominous",
        "weak", "fragile", "vulnerable", "precarious", "strained",
        "stressed", "troubled", "distressed", "declining", "deteriorating",
    }
}

tech_res, blue_chip_res = calc_tech_and_blue_chip_sent(model, blue_chip_dataloader, tech_dataloader, ending, sentiment_verb)
tech_line, blue_chip_line, result_dict = calculate_mean_polarities(tech_res, blue_chip_res)

In [ ]:
ending = "Our outlook for the next fiscal year is fundamentally"

outlook_verbalizer = {
    "positive": {
        "strong", "robust", "healthy", "positive", "favorable",
        "promising", "bright", "optimistic", "bullish", "upbeat",
        "encouraging", "solid", "stable", "resilient",
        "improving", "strengthening", "recovering", "expanding",
        "growing", "accelerating", "thriving",
        "competitive", "advantaged", "positioned", "strategic"
    },
    "negative": {
        "weak", "poor", "challenging", "difficult", "negative",
        "unfavorable", "pessimistic", "bearish", "concerning",
        "troubling", "precarious", "fragile", "vulnerable",
        "declining", "deteriorating", "weakening", "contracting",
        "shrinking", "stagnant", "flat",
        "uncertain", "unpredictable", "volatile", "risky", "exposed",
        "cautious", "guarded", "prudent", "conservative", "measured",
        "tempered", "qualified", "conditional",
        "variable", "dependent", "speculative", "worthless"
    }
}

tech_res, blue_chip_res = calc_tech_and_blue_chip_sent(model, blue_chip_dataloader, tech_dataloader, ending, outlook_verbalizer)
tech_line, blue_chip_line, result_dict = calculate_mean_polarities(tech_res, blue_chip_res)

In [ ]:
ending = "In one word, our forward-looking statements are"

tech_res, blue_chip_res = calc_tech_and_blue_chip_sent(model, blue_chip_dataloader, tech_dataloader, ending, outlook_verbalizer)
tech_line, blue_chip_line, result_dict = calculate_mean_polarities(tech_res, blue_chip_res)

In [ ]:
ending = "Based on current trends, we view the company's near-term trajectory with"

trajectory_sentiment_verbalizer = {
    "positive": {
        "optimism", "confidence", "hope", "positivity", "enthusiasm",
        "excitement", "assurance", "conviction", "faith", "encouragement",
        "satisfaction", "comfort", "approval", "favorability", "buoyancy",
        "reassurance", "certainty", "conviction", "approval", "endorsement"
    },
    
    "negative": {
        "concern", "worry", "pessimism", "anxiety", "apprehension",
        "caution", "doubt", "skepticism", "unease", "trepidation",
        "alarm", "disappointment", "frustration", "discouragement",
        "disapproval", "dismay", "despondency", "wariness", "hesitation",
        "suspicion", "nervousness", "foreboding", "disquiet"
    }
}

tech_res, blue_chip_res = calc_tech_and_blue_chip_sent(model, blue_chip_dataloader, tech_dataloader, ending, trajectory_sentiment_verbalizer)
tech_line, blue_chip_line, result_dict = calculate_mean_polarities(tech_res, blue_chip_res)

In [ ]:
ending = "Based on current trends, we view the company's long-term trajectory with"

tech_res, blue_chip_res = calc_tech_and_blue_chip_sent(model, blue_chip_dataloader, tech_dataloader, ending, trajectory_sentiment_verbalizer)
tech_line, blue_chip_line, result_dict = calculate_mean_polarities(tech_res, blue_chip_res)